# 2단계 · CosFace 탐침

같은 RUN_ID를 사용합니다. [실행과 결과 안내](HOW_TO_READ.md)

In [ ]:
# cell 1 : 설정 및 경로 확인
import json
import os
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "fine-tune/modules").is_dir())
sys.path.insert(0, str(repo / "fine-tune"))
from modules.clean_training import final_artifact, recipe, run_directory, run_stage
from modules.clean_campaign import evaluation_dir, model_dir, origin_matrix, parent_validation, read_result, run_full, run_probe, validate_choices

DATA_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets"))
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models"))
RUN_ID = "carbon_mfn1024_s8972_v2"
SEED = 8972
EXECUTE = False
RESUME = False
KEEP_RAW_RESULTS = False
PARENT_SOURCE = "clean_parent"
PROBE_DATASETS = ["tinyface", "survface"]
run = run_directory(MODEL_ROOT, RUN_ID)
validate_choices(PROBE_DATASETS, ["cosface"], "paper_table1", PARENT_SOURCE)
paths = [model_dir(run, "paper_table1", PARENT_SOURCE, ds, "cosface") for ds in PROBE_DATASETS]
display(pd.DataFrame({"dataset": PROBE_DATASETS, "path": paths}))

In [ ]:
# cell 2 : 실행
if EXECUTE:
    probe = run_probe(DATA_ROOT, run, PARENT_SOURCE, PROBE_DATASETS,
                      seed=SEED, keep_raw_results=KEEP_RAW_RESULTS, resume=RESUME)

In [ ]:
# cell 3 : 결과 확인
rows = []
for ds in PROBE_DATASETS:
    for source in ("clean_parent", "legacy_parent_wd0"):
        p = evaluation_dir(model_dir(run, "paper_table1", source, ds, "cosface"), ds, probe=True)
        if (p / "COMPLETED").is_file():
            summary = read_result(p)
            rows.extend(dict(parent_source=source, parent_sha256=summary["parent_sha256"], **row)
                        for row in summary["comparison"])
if rows:
    display(pd.DataFrame(rows))
else:
    display(pd.DataFrame([dict(dataset=ds, training=(p / "COMPLETED").is_file(),
                               evaluation=(evaluation_dir(p, ds, probe=True) / "COMPLETED").is_file())
                          for ds, p in zip(PROBE_DATASETS, paths, strict=True)]))